In [ ]:
import torch
import matplotlib.pyplot as plt
import random

## 回调


### 回调作为 GUI 事件


In [ ]:
import ipywidgets as widgets

来自 [ipywidget 文档](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20Events.html)：

- *按钮组件用于处理鼠标点击事件。可以使用 Button 的 on_click 方法注册一个函数，当按钮被点击时该函数将被调用*


In [ ]:
w = widgets.Button(description='Click me')

In [ ]:
w

Button(description='Click me', style=ButtonStyle())

In [ ]:
def f(o): print('hi')

In [ ]:
w.on_click(f)

*注意：当回调以这种方式使用时，它们通常被称为"事件"。*


### 创建您自己的回调


In [ ]:
from time import sleep

In [ ]:
def slow_calculation():
    res = 0
    for i in range(5):
        res += i*i
        sleep(1)
    return res

In [ ]:
slow_calculation()

30

In [ ]:
def slow_calculation(cb=None):
    res = 0
    for i in range(5):
        res += i*i
        sleep(1)
        if cb: cb(i)
    return res

In [ ]:
def show_progress(epoch): print(f"Awesome! We've finished epoch {epoch}!")

In [ ]:
slow_calculation(show_progress)

Awesome! We've finished epoch 0!
Awesome! We've finished epoch 1!
Awesome! We've finished epoch 2!
Awesome! We've finished epoch 3!
Awesome! We've finished epoch 4!


30

### Lambda 函数与偏函数


In [ ]:
slow_calculation(lambda o: print(f"Awesome! We've finished epoch {o}!"))

Awesome! We've finished epoch 0!
Awesome! We've finished epoch 1!
Awesome! We've finished epoch 2!
Awesome! We've finished epoch 3!
Awesome! We've finished epoch 4!


30

In [ ]:
def show_progress(exclamation, epoch): print(f"{exclamation}! We've finished epoch {epoch}!")

In [ ]:
slow_calculation(lambda o: show_progress("OK I guess", o))

OK I guess! We've finished epoch 0!
OK I guess! We've finished epoch 1!
OK I guess! We've finished epoch 2!
OK I guess! We've finished epoch 3!
OK I guess! We've finished epoch 4!


30

In [ ]:
def make_show_progress(exclamation):
    def _inner(epoch): print(f"{exclamation}! We've finished epoch {epoch}!")
    return _inner

In [ ]:
slow_calculation(make_show_progress("Nice!"))

Nice!! We've finished epoch 0!
Nice!! We've finished epoch 1!
Nice!! We've finished epoch 2!
Nice!! We've finished epoch 3!
Nice!! We've finished epoch 4!


30

In [ ]:
from functools import partial

In [ ]:
slow_calculation(partial(show_progress, "OK I guess"))

OK I guess! We've finished epoch 0!
OK I guess! We've finished epoch 1!
OK I guess! We've finished epoch 2!
OK I guess! We've finished epoch 3!
OK I guess! We've finished epoch 4!


30

In [ ]:
f2 = partial(show_progress, "OK I guess")

### 回调作为可调用类


In [ ]:
class ProgressShowingCallback():
    def __init__(self, exclamation="Awesome"): self.exclamation = exclamation
    def __call__(self, epoch): print(f"{self.exclamation}! We've finished epoch {epoch}!")

In [ ]:
cb = ProgressShowingCallback("Just super")

In [ ]:
slow_calculation(cb)

Just super! We've finished epoch 0!
Just super! We've finished epoch 1!
Just super! We've finished epoch 2!
Just super! We've finished epoch 3!
Just super! We've finished epoch 4!


30

### 多个回调函数；`*args` 和 `**kwargs`


In [ ]:
def f(*a, **b): print(f"args: {a}; kwargs: {b}")

In [ ]:
f(3, 'a', thing1="hello")

args: (3, 'a'); kwargs: {'thing1': 'hello'}


In [ ]:
def g(a,b,c=0): print(a,b,c)

In [ ]:
args = [1,2]
kwargs = {'c':3}
g(*args, **kwargs)

1 2 3


In [ ]:
def slow_calculation(cb=None):
    res = 0
    for i in range(5):
        if cb: cb.before_calc(i)
        res += i*i
        sleep(1)
        if cb: cb.after_calc(i, val=res)
    return res

In [ ]:
class PrintStepCallback():
    def before_calc(self, *args, **kwargs): print(f"About to start")
    def after_calc (self, *args, **kwargs): print(f"Done step")

In [ ]:
slow_calculation(PrintStepCallback())

About to start
Done step
About to start
Done step
About to start
Done step
About to start
Done step
About to start
Done step


30

In [ ]:
class PrintStatusCallback():
    def __init__(self): pass
    def before_calc(self, epoch, **kwargs): print(f"About to start: {epoch}")
    def after_calc (self, epoch, val, **kwargs): print(f"After {epoch}: {val}")

In [ ]:
slow_calculation(PrintStatusCallback())

About to start: 0
After 0: 0
About to start: 1
After 1: 1
About to start: 2
After 2: 5
About to start: 3
After 3: 14
About to start: 4
After 4: 30


30

### 修改行为


In [ ]:
def slow_calculation(cb=None):
    res = 0
    for i in range(5):
        if cb and hasattr(cb,'before_calc'): cb.before_calc(i)
        res += i*i
        sleep(1)
        if cb and hasattr(cb,'after_calc'):
            if cb.after_calc(i, res):
                print("stopping early")
                break
    return res

In [ ]:
class PrintAfterCallback():
    def after_calc (self, epoch, val):
        print(f"After {epoch}: {val}")
        if val>10: return True

In [ ]:
slow_calculation(PrintAfterCallback())

After 0: 0
After 1: 1
After 2: 5
After 3: 14
stopping early


14

In [ ]:
class SlowCalculator():
    def __init__(self, cb=None): self.cb,self.res = cb,0
    
    def callback(self, cb_name, *args):
        if not self.cb: return
        cb = getattr(self.cb,cb_name, None)
        if cb: return cb(self, *args)

    def calc(self):
        for i in range(5):
            self.callback('before_calc', i)
            self.res += i*i
            sleep(1)
            if self.callback('after_calc', i):
                print("stopping early")
                break

In [ ]:
class ModifyingCallback():
    def after_calc (self, calc, epoch):
        print(f"After {epoch}: {calc.res}")
        if calc.res>10: return True
        if calc.res<3: calc.res = calc.res*2

In [ ]:
calculator = SlowCalculator(ModifyingCallback())

In [ ]:
calculator.calc()
calculator.res

After 0: 0
After 1: 1
After 2: 6
After 3: 15
stopping early


15

## `__dunder__` 相关内容


任何看起来像 `__this__` 的东西，在某种程度上都是<em>特殊的</em>。Python 或某些库可以定义一些函数，并在特定的时机自动调用它们。例如，当你的类正在初始化一个新对象时，Python 会调用 `__init__`。这些都是 Python [数据模型](https://docs.python.org/3/reference/datamodel.html#object.__init__) 的一部分。

举个例子，当 Python 遇到 `+` 时，它会调用特殊方法 `__add__`。如果你尝试在 Jupyter（或 Python 中的许多其他地方）中显示一个对象，它会调用 `__repr__`。


In [ ]:
class SloppyAdder():
    def __init__(self,o): self.o=o
    def __add__(self,b): return SloppyAdder(self.o + b.o + 0.01)
    def __repr__(self): return str(self.o)

In [ ]:
a = SloppyAdder(1)
b = SloppyAdder(2)
a+b

3.01

你可能需要了解的特殊方法（请参阅上方数据模型链接）包括：

- `__getitem__`
- `__getattr__`
- `__setattr__`
- `__del__`
- `__init__`
- `__new__`
- `__enter__`
- `__exit__`
- `__len__`
- `__repr__`
- `__str__`


### `__getattr__` 和 `getattr`


In [ ]:
class A: a,b=1,2

In [ ]:
a = A()

In [ ]:
a.b

2

In [ ]:
getattr(a, 'b')

2

In [ ]:
getattr(a, 'b' if random.random()>0.5 else 'a')

2

In [ ]:
class B:
    a,b=1,2
    def __getattr__(self, k):
        if k[0]=='_': raise AttributeError(k)
        return f'Hello from {k}'

In [ ]:
b = B()

In [ ]:
b.a

1

In [ ]:
b.foo

'Hello from foo'

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文档已使用 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言的原始文档应被视为权威来源。对于重要信息，建议使用专业人工翻译。对于因使用本翻译而产生的任何误解或误读，我们概不负责。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
